In [ ]:
from google import genai
import os
api_key=os.environ['GOOGLE_API_KEY']
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

In [4]:
# Short-term memory (verbatim, recent turns)
recent_messages = []

# Long-term memory (compressed summary)
conversation_summary = ""

# Structured ticket state (machine-readable)
ticket_state = {
    "issue": None,
    "status": "open",
    "attempted_fixes": [],
}

In [ ]:
SYSTEM_INSTRUCTION = """
You are a customer support assistant helping resolve a technical issue.

Goals:
- Understand the user's issue
- Avoid repeating questions that were already answered
- Do not suggest fixes that were already attempted
- Move the ticket toward resolution

You are given:
- A summary of earlier conversation (long-term memory)
- Current ticket state (structured)
- Recent messages (short-term memory)

Rules:
- Do NOT suggest fixes that were already attempted.
- If an issue is resolved, acknowledge it and do not repeat troubleshooting.
- If the same question is asked again, answer differently based on ticket state.
- If new information contradicts a resolved state, reopen the ticket.
- Be concise and professional.
"""


In [13]:
def build_context():
    return f"""
=== Ticket Summary (Long-Term Memory) ===
{conversation_summary or "No summary yet."}

=== Ticket State (Structured Memory) ===
Issue: {ticket_state["issue"]}
Status: {ticket_state["status"]}
Attempted Fixes: {ticket_state["attempted_fixes"]}

=== Recent Conversation (Short-Term Memory) ===
{recent_messages[-4:]}
"""

In [ ]:
def update_ticket_state_from_user(user_message: str):
    global ticket_state

    prompt = f"""
        Analyze the user message and update ticket state if needed.

        User message:
        {user_message}

        Current ticket state:
        {ticket_state}

        Return JSON ONLY with fields to update.
        Possible fields:
        - issue
        - status
        - attempted_fixes (append new ones)
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    try:
        updates = eval(response.text)
        for key, value in updates.items():
            if key == "attempted_fixes" and isinstance(value, list):
                for fix in value:
                    if fix not in ticket_state["attempted_fixes"]:
                        ticket_state["attempted_fixes"].append(fix)
            else:
                ticket_state[key] = value
    except Exception:
        pass


In [ ]:
def update_conversation_summary():
    global conversation_summary

    if not recent_messages:
        return

    prompt = f"""
        Summarize this support conversation.
        Focus on:
        - the original issue
        - fixes attempted
        - whether the issue was resolved

        Conversation:
        {recent_messages}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    conversation_summary = response.text.strip()


In [ ]:
def update_ticket_state(user_message):
    global ticket_state

    prompt = f"""
        Given the user message below, update ticket fields if applicable.

        User message:
        {user_message}

        Current ticket state:
        {ticket_state}

        Return JSON with any updates, or empty JSON if none.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
    )

    try:
        updates = eval(response.text)
        for key, value in updates.items():
            ticket_state[key] = value
    except Exception:
        pass


In [9]:
def assistant_reply(user_message: str):
    context = build_context()

    response = client.models.generate_content(
        model=MODEL,
        contents=[
            SYSTEM_INSTRUCTION,
            context,
            f"User message: {user_message}",
        ],
    )

    return response.text.strip()


In [ ]:
print("\nContext-Aware Support Ticket Assistant (Initial Session)")
print("Describe your issue. Type 'fixed' once the issue is resolved.\n")

while True:
    user_input = input("User: ")
    if user_input.lower() == "quit":
        break

    recent_messages.append(f"User: {user_input}")

    update_ticket_state_from_user(user_input)

    reply = assistant_reply(user_input)
    print("\nAssistant:", reply)
    print("→ Type 'fixed' if the issue is resolved.\n")

    recent_messages.append(f"Assistant: {reply}")

    if "works now" in user_input.lower() or "fixed" in user_input.lower():
        ticket_state["status"] = "resolved"
        update_conversation_summary()
        recent_messages.clear()
        print("\nTicket marked as resolved.\n")
        break



🎫 Context-Aware Support Ticket Assistant (Initial Session)
Describe your issue. Type 'fixed' once the issue is resolved.


Assistant: I understand you're having trouble logging into your account.

Could you please describe what happens when you try to log in? Are you seeing an error message, or is something else occurring?
→ Type 'fixed' if the issue is resolved.


Assistant: Thanks for providing that detail. "Invalid credentials" usually means there's a mismatch between the username and password entered.

Could you please double-check your username and password for any typos? If you're certain they're correct, the next step would be to try resetting your password. Would you like me to guide you through that process?
→ Type 'fixed' if the issue is resolved.


Assistant: Great to hear your password reset resolved the login issue! I'll go ahead and close this ticket. Please reach out if you have any other questions.
→ Type 'fixed' if the issue is resolved.



ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 45.806179135s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'

Note:
This example assumes the user is reopening an existing ticket.
In real systems, ticket routing and creation logic would live outside the model.
The goal here is to demonstrate how persisted context changes assistant behavior.

In [ ]:
print("\nContext-Aware Support Ticket Assistant (Reopening Ticket)")
print("The previous ticket is being reopened.\n")

# Explicit reopen (no ambiguity)
ticket_state["status"] = "open"

while True:
    user_input = input("User: ")
    if user_input.lower() == "quit":
        break

    recent_messages.append(f"User: {user_input}")

    update_ticket_state_from_user(user_input)

    reply = assistant_reply(user_input)
    print("\nAssistant:", reply)
    print("→ Type 'fixed' if the issue is resolved.\n")

    recent_messages.append(f"Assistant: {reply}")

    if "works now" in user_input.lower() or "fixed" in user_input.lower():
        ticket_state["status"] = "resolved"
        update_conversation_summary()
        recent_messages.clear()
        print("\nTicket re-resolved.\n")
        break



🎫 Context-Aware Support Ticket Assistant (Reopening Ticket)
The previous ticket is being reopened.


Assistant: I'm sorry to hear the issue has returned. I've reopened your ticket.

Could you please describe what happens when you try to log in this morning? Are you seeing the same "invalid credentials" message, or something different?
→ Type 'fixed' if the issue is resolved.


Assistant: Thank you for clarifying. A "system ran into an error, please try again" message indicates a different issue than invalid credentials.

Could you please tell me:
1.  Does this error occur every time you attempt to log in?
2.  Are you trying to log in from the same device and browser you used previously?
→ Type 'fixed' if the issue is resolved.


Assistant: Thank you for confirming.

To help us investigate this "system ran into an error" message further, could you please tell me:
1. What specific browser (e.g., Chrome, Firefox, Edge) and version are you using?
2. What operating system is your device ru